# Dynamic Tables: Always-Fresh Feature Engineering

This notebook demonstrates how Dynamic Tables solve the **stale feature** problem:  
source tables update, and your downstream feature store + aggregations **automatically refresh** — no orchestrator, no manual intervention.

### Pipeline Architecture

```
  credit_bureau_scores  ──┐
                          │
  transaction_history  ───┼──▶  feature_store (DT, 1-min lag)  ──▶  acquisition_summary (DT, downstream)
                          │
  account_demographics ───┘
```

| Stage | What It Does | Target Lag |
|-------|-------------|------------|
| **Raw Sources** | Credit bureau, transactions, demographics | — (base tables) |
| **Feature Store** | Joins, cleans, derives risk tiers + income bands | 1 minute |
| **Acquisition Summary** | Portfolio-level metrics by segment | Downstream (auto) |

---
## 1. Setup: Database, Schema, and Raw Source Tables

We create three source tables representing upstream data feeds that a valuations model would consume.

In [ ]:
%%sql -r setup_db
CREATE DATABASE IF NOT EXISTS demo_ian;
CREATE SCHEMA IF NOT EXISTS demo_ian.valuations;

In [ ]:
%%sql -r create_credit_bureau
CREATE OR REPLACE TABLE demo_ian.valuations.credit_bureau_scores (
    applicant_id    VARCHAR(20),
    fico_score      INT,
    bureau_update_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

INSERT INTO demo_ian.valuations.credit_bureau_scores (applicant_id, fico_score)
VALUES
    ('APP-001', 785), ('APP-002', 720), ('APP-003', 650),
    ('APP-004', 810), ('APP-005', 695), ('APP-006', 740),
    ('APP-007', 590), ('APP-008', 760), ('APP-009', 830),
    ('APP-010', 670), ('APP-011', 755), ('APP-012', 620),
    ('APP-013', 800), ('APP-014', 710), ('APP-015', 680);

In [ ]:
%%sql -r create_transactions
CREATE OR REPLACE TABLE demo_ian.valuations.transaction_history (
    applicant_id VARCHAR(20),
    txn_amount   DECIMAL(10,2),
    txn_category VARCHAR(30),
    txn_date     DATE
);

INSERT INTO demo_ian.valuations.transaction_history
SELECT
    'APP-' || LPAD(seq4() % 15 + 1, 3, '0'),
    ROUND(UNIFORM(5.00, 500.00, RANDOM()), 2),
    CASE UNIFORM(1, 5, RANDOM())
        WHEN 1 THEN 'Travel'
        WHEN 2 THEN 'Dining'
        WHEN 3 THEN 'Grocery'
        WHEN 4 THEN 'Entertainment'
        ELSE 'Other'
    END,
    DATEADD('day', -UNIFORM(1, 90, RANDOM()), CURRENT_DATE())
FROM TABLE(GENERATOR(ROWCOUNT => 300));

In [ ]:
%%sql -r create_demographics
CREATE OR REPLACE TABLE demo_ian.valuations.account_demographics (
    applicant_id      VARCHAR(20),
    annual_income     INT,
    years_with_bank   INT,
    existing_products INT,
    product_requested VARCHAR(30)
);

INSERT INTO demo_ian.valuations.account_demographics
VALUES
    ('APP-001', 185000, 8,  3, 'Venture X'),
    ('APP-002', 120000, 5,  2, 'Savor One'),
    ('APP-003',  65000, 2,  0, 'Quicksilver'),
    ('APP-004', 250000, 12, 4, 'Venture X'),
    ('APP-005',  88000, 3,  1, 'Savor'),
    ('APP-006', 145000, 7,  2, 'Venture X'),
    ('APP-007',  52000, 1,  0, 'Quicksilver'),
    ('APP-008', 175000, 6,  3, 'Savor'),
    ('APP-009', 310000, 15, 5, 'Venture X'),
    ('APP-010',  78000, 4,  1, 'Savor One'),
    ('APP-011', 160000, 9,  2, 'Venture X'),
    ('APP-012',  58000, 1,  0, 'Quicksilver'),
    ('APP-013', 220000, 11, 4, 'Venture X'),
    ('APP-014', 105000, 6,  1, 'Savor One'),
    ('APP-015',  72000, 3,  1, 'Quicksilver');

---
## 2. Dynamic Table Stage 1: Feature Store

This is the core of the pipeline. It joins across all three source tables, computes derived features (risk tiers, income bands, spend aggregates, cross-sell flags), and produces a clean feature vector.

**Target lag: 1 minute** — when any source table updates, this DT refreshes within 60 seconds. No orchestrator needed.

In [ ]:
%%sql -r create_feature_store
CREATE OR REPLACE DYNAMIC TABLE demo_ian.valuations.feature_store
    TARGET_LAG = '1 minute'
    WAREHOUSE = DASH_S
AS
SELECT
    cb.applicant_id,
    cb.fico_score,
    CASE
        WHEN cb.fico_score >= 740 THEN 'Prime'
        WHEN cb.fico_score >= 670 THEN 'Near-Prime'
        ELSE 'Sub-Prime'
    END AS risk_tier,

    ad.annual_income,
    CASE
        WHEN ad.annual_income >= 150000 THEN 'High'
        WHEN ad.annual_income >= 75000  THEN 'Medium'
        ELSE 'Standard'
    END AS income_band,

    ad.years_with_bank,
    ad.existing_products,
    ad.product_requested,

    COALESCE(t.avg_monthly_spend, 0)  AS avg_monthly_spend,
    COALESCE(t.txn_count_90d, 0)      AS txn_count_90d,
    COALESCE(t.top_category, 'None')   AS top_spend_category,

    CASE
        WHEN ad.existing_products > 0 THEN 'Cross-Sell'
        ELSE 'New-to-Bank'
    END AS acquisition_type

FROM demo_ian.valuations.credit_bureau_scores cb
JOIN demo_ian.valuations.account_demographics ad
    ON cb.applicant_id = ad.applicant_id
LEFT JOIN (
    SELECT
        applicant_id,
        ROUND(AVG(txn_amount), 2)  AS avg_monthly_spend,
        COUNT(*)                   AS txn_count_90d,
        MODE(txn_category)         AS top_category
    FROM demo_ian.valuations.transaction_history
    WHERE txn_date >= DATEADD('day', -90, CURRENT_DATE())
    GROUP BY applicant_id
) t ON cb.applicant_id = t.applicant_id;

---
## 3. Dynamic Table Stage 2: Acquisition Summary

Portfolio-level metrics aggregated by product, risk tier, and acquisition type. This is what a Business Director looks at — not individual applications, but segment-level performance.

**Target lag: DOWNSTREAM** — automatically refreshes whenever the feature store updates. Zero configuration.

In [ ]:
%%sql -r create_acq_summary
CREATE OR REPLACE DYNAMIC TABLE demo_ian.valuations.acquisition_summary
    TARGET_LAG = DOWNSTREAM
    WAREHOUSE = DASH_S
AS
SELECT
    product_requested,
    risk_tier,
    income_band,
    acquisition_type,
    COUNT(*)                          AS applicant_count,
    ROUND(AVG(fico_score), 0)         AS avg_fico,
    ROUND(AVG(annual_income), 0)      AS avg_income,
    ROUND(AVG(avg_monthly_spend), 2)  AS avg_spend,
    ROUND(AVG(txn_count_90d), 0)      AS avg_txn_count
FROM demo_ian.valuations.feature_store
GROUP BY product_requested, risk_tier, income_band, acquisition_type;

---
## 4. Verify the Pipeline

Let's query both dynamic tables to confirm they're populated and see the current state.

In [ ]:
%%sql -r view_features
SELECT * FROM demo_ian.valuations.feature_store
ORDER BY applicant_id

In [ ]:
%%sql -r view_summary
SELECT * FROM demo_ian.valuations.acquisition_summary
ORDER BY product_requested, risk_tier

---
## 5. Source Update: Watch the Pipeline Refresh Itself

Now the important part. We insert new applicants into the **source tables**. We do NOT touch the dynamic tables.

Within 1 minute, `feature_store` refreshes on its own, and `acquisition_summary` follows automatically because its lag is `DOWNSTREAM`.

**In your world today**: this would need a manual refresh or a wait for the nightly batch.

Run the cell below to start the clock — then we'll look at some notebook features while the refresh happens in the background.

In [ ]:
%%sql -r insert_new_data
-- New high-value applicants come in
INSERT INTO demo_ian.valuations.credit_bureau_scores (applicant_id, fico_score)
VALUES ('APP-100', 820), ('APP-101', 690), ('APP-102', 775);

INSERT INTO demo_ian.valuations.account_demographics
VALUES
    ('APP-100', 275000, 10, 3, 'Venture X'),
    ('APP-101',  82000, 2,  0, 'Quicksilver'),
    ('APP-102', 165000, 7,  2, 'Savor');

INSERT INTO demo_ian.valuations.transaction_history
VALUES
    ('APP-100', 450.00, 'Travel',        CURRENT_DATE()),
    ('APP-100', 120.00, 'Dining',         DATEADD('day', -5, CURRENT_DATE())),
    ('APP-101',  35.00, 'Grocery',        CURRENT_DATE()),
    ('APP-102', 280.00, 'Entertainment',  CURRENT_DATE());

---
## While That Refreshes: Notebook Features Worth Noting

The insert is done and the dynamic tables are refreshing in the background. While we wait, here's what's different about this notebook experience.

### Native SQL cells

Every SQL cell in this notebook is a **first-class SQL cell** — not a `%%sql` magic command or a `session.sql()` wrapper. SQL is a dedicated cell type alongside Python and Markdown.

### Cell referencing: SQL results flow into Python (and other SQL cells)

The `view_features` query from earlier stored its result as a variable. We can reference it directly in Python — no `cursor.fetchall()`, no boilerplate:

In [ ]:
# view_features is the result from the SQL cell above — available directly
# No cursor.fetchall(), no pd.read_sql(), no boilerplate
import pandas as pd

df = view_features if isinstance(view_features, pd.DataFrame) else view_features.to_pandas()

print(f"Feature store has {len(df)} applicants")
print(f"\nRisk tier distribution:")
print(df['RISK_TIER'].value_counts().to_string())
print(f"\nProducts requested:")
print(df['PRODUCT_REQUESTED'].value_counts().to_string())

SQL cells can also reference other SQL cells using Jinja templating — `{{variable_name}}`:

In [ ]:
%%sql -r sql_references_sql
-- Reference the feature_store SQL result directly with Jinja
-- No need to re-query the table — reuse the result from the earlier cell
SELECT
    risk_tier,
    acquisition_type,
    COUNT(*) AS count,
    ROUND(AVG(fico_score), 0) AS avg_fico,
    ROUND(AVG(annual_income), 0) AS avg_income
FROM {{view_features}}
GROUP BY risk_tier, acquisition_type
ORDER BY risk_tier, acquisition_type

### Python variables flow into SQL too

Define a filter in Python, use it in SQL — no f-string concatenation (which is an injection risk):

In [ ]:
target_product = 'Venture X'
min_fico = 740

In [ ]:
%%sql -r filtered_query
-- Python variables injected via Jinja — no f-strings, no injection risk
SELECT applicant_id, fico_score, annual_income, risk_tier
FROM demo_ian.valuations.feature_store
WHERE product_requested = '{{target_product}}'
  AND fico_score >= {{min_fico}}
ORDER BY fico_score DESC

### What else is different

| Feature | Jupyter on EMP | Snowflake Notebooks |
|---------|---------------|---------------------|
| **Compute** | Local kernel — large queries can OOM | SQL + Snowpark push down to the warehouse. Kernel handles Python logic only. |
| **Startup** | 10-15 min (provisions infra) | Seconds (warehouse is shared infrastructure) |
| **Persistence** | Deleted after 14 days | First-class Snowflake object, persists until dropped |
| **Idle timeout** | Fixed | Configurable up to 72 hours. Navigate away and come back — session state preserved. |
| **Multi-user** | Shared kernel risk | Each user gets an independent session on the same notebook |
| **Scheduling** | External scheduler | Built-in — schedule a notebook to run on a cron directly from the UI |
| **Git** | CLI or GitHub Desktop drag-and-drop | Native git integration in Workspaces |
| **Packages** | pip/conda in the kernel, mixed environments | Packages panel in the UI, or pip against curated Anaconda / your Artifactory |
| **Visualization** | Same | matplotlib, altair, plotly all supported |

---

---
## 6. Back to the Pipeline: Did It Refresh?

That took about a minute. Let's check whether the dynamic tables picked up the new applicants — without us doing anything.

In [ ]:
SELECT
    name,
    state,
    state_message,
    refresh_start_time,
    refresh_end_time
FROM TABLE(DEMO_IAN.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
    NAME_PREFIX => 'DEMO_IAN.VALUATIONS.'
))
ORDER BY refresh_start_time DESC
LIMIT 10

In [ ]:
%%sql -r verify_new_applicants
-- The new applicants should appear here without any manual refresh
SELECT * FROM demo_ian.valuations.feature_store
WHERE applicant_id IN ('APP-100', 'APP-101', 'APP-102')

In [ ]:
%%sql -r verify_summary_updated
-- The downstream summary also updated automatically
SELECT * FROM demo_ian.valuations.acquisition_summary
ORDER BY product_requested, risk_tier

**Tip**: You can also visualize this pipeline in Snowsight — go to **Monitoring > Dynamic Tables** to see the full DAG, refresh history, and lag status.

---

## 7. Dependency Management: The Cyber Team Scenario

**Scenario**: Your cyber team notifies you that `numpy 1.26.4` has a CVE. You need to downgrade to `numpy==1.24.3`.

Two problems to solve:
1. **Resolution**: What versions of your other 15 packages are compatible with the new numpy?
2. **Blast radius**: Which notebooks, UDFs, and procedures in your account use the flagged package?

### Problem 1: Automatic dependency resolution

When you change one package version, pip resolves all other packages to compatible versions automatically. No trial and error.

**Important**: By default, the notebook environment is **locked down** — pip can only install from pre-installed packages or a configured artifact repository. It does NOT reach out to the public internet. This is security by design.

In [ ]:
# What's currently installed — ~100 pre-installed data science packages
!pip list 2>/dev/null | head -25

In [ ]:
# Try upgrading numpy to the latest available version
# pip resolves ALL other package versions automatically
# --dry-run shows what WOULD change without actually installing
!pip install numpy==2.4.6 --dry-run 2>&1 | tail -20

**What just happened**:

- `--dry-run` shows what pip *would* install without actually changing anything
- If you saw a "Package sources not available" warning, that's the point — **the environment is locked down by default**. pip only resolves from pre-installed packages or your configured artifact repository. No unvetted internet access.
- Once your Artifactory is connected (see Section 7), `pip install numpy==<approved-version>` resolves against YOUR approved packages only — and it handles the cascading version changes for all other packages automatically

**The workflow with Artifactory connected**:
1. Cyber team flags `numpy 1.26.4` with a CVE
2. Your team approves `numpy 2.4.6` in Artifactory
3. In the notebook: `pip install numpy==2.4.6` — pip resolves all 15 other packages to compatible versions
4. No trial and error. No manual cross-referencing.

### Problem 2: Blast radius — what's affected?

In [ ]:
%%sql -r blast_radius
-- One query: every UDF and procedure in the account that depends on numpy
-- Includes TRANSITIVE dependencies (not just what's declared)
SELECT
    'FUNCTION' AS object_type,
    function_catalog AS database_name,
    function_schema AS schema_name,
    function_name AS object_name,
    packages AS declared_packages,
    installed_packages
FROM snowflake.account_usage.functions
WHERE deleted IS NULL
    AND function_language = 'PYTHON'
    AND installed_packages ILIKE '%numpy%'

UNION ALL

SELECT
    'PROCEDURE',
    procedure_catalog,
    procedure_schema,
    procedure_name,
    packages,
    installed_packages
FROM snowflake.account_usage.procedures
WHERE deleted IS NULL
    AND procedure_language = 'PYTHON'
    AND installed_packages ILIKE '%numpy%'

ORDER BY database_name, schema_name, object_name

**What this solves**: When your cyber team flags a package, you now have:
1. **Automatic resolution** of cascading version changes (pip handles it)
2. **One query** to find every affected object in the account (no manual hunting)
3. **Package retention** — deployed objects keep working on pinned versions until you deliberately update them

---

## 8. Git Integration & Private Repository

This notebook was pulled from a git repository — no GitHub Desktop drag-and-drop, no CLI gymnastics. Git integration is native in Snowflake.

### Connecting to a private repository

In [ ]:
%%sql -r git_setup
-- Step 1: Store your PAT securely
-- CREATE SECRET demo_ian.valuations.git_secret
--     TYPE = PASSWORD
--     USERNAME = 'your-github-username'
--     PASSWORD = '<your-personal-access-token>';

-- Step 2: Create an API integration for GitHub
-- CREATE API INTEGRATION git_api_integration
--     API_PROVIDER = GIT_HTTPS_API
--     API_ALLOWED_PREFIXES = ('https://github.com/your-org')
--     ALLOWED_AUTHENTICATION_SECRETS = (demo_ian.valuations.git_secret)
--     ENABLED = TRUE;

-- Step 3: Register the repository
-- CREATE GIT REPOSITORY demo_ian.valuations.my_repo
--     API_INTEGRATION = git_api_integration
--     GIT_CREDENTIALS = demo_ian.valuations.git_secret
--     ORIGIN = 'https://github.com/your-org/valuations-notebooks.git';

-- That's it. 3 SQL statements to connect a private repo.
SELECT 'Git integration: 3 SQL statements, no CLI, no SSH keys, no GitHub Desktop' AS status

### Connecting to a private package repository (Artifactory)

For organizations that maintain their own Python package repository (JFrog Artifactory, Nexus, etc.), Snowflake can pull packages directly from it. Every `pip install` in every notebook resolves against your approved packages.

In [ ]:
%%sql -r artifactory_setup
-- Step 1: Store Artifactory credentials
-- CREATE SECRET demo_ian.valuations.artifactory_secret
--     TYPE = PASSWORD
--     USERNAME = 'svc-snowflake'
--     PASSWORD = '<artifactory-token>';

-- Step 2: API integration (supports PrivateLink for internal traffic)
-- CREATE API INTEGRATION artifactory_integration
--     API_PROVIDER = ARTIFACT_REPOSITORY_API
--     API_ALLOWED_PREFIXES = ('https://artifactory.internal.yourcompany.com')
--     ALLOWED_AUTHENTICATION_SECRETS = (demo_ian.valuations.artifactory_secret)
--     ENABLED = TRUE;

-- Step 3: Register the artifact repository
-- CREATE ARTIFACT REPOSITORY demo_ian.valuations.internal_pypi
--     TYPE = PYPI
--     API_INTEGRATION = artifactory_integration
--     INDEX_URL = 'https://artifactory.internal.yourcompany.com/pypi/simple/'
--     AUTHENTICATION_SECRET = demo_ian.valuations.artifactory_secret;

-- Once set as default, ALL notebooks pull from it automatically:
-- ALTER ACCOUNT SET DEFAULT_PYTHON_ARTIFACT_REPOSITORY
--     = demo_ian.valuations.internal_pypi;

SELECT 'Artifactory: governance stays in your existing tooling' AS status

**Key points for your security team**:
- Packages are pulled from YOUR Artifactory — your allowlists, quarantine rules, and version pins apply
- Supports **PrivateLink** — traffic never leaves your VPC
- **Package retention** ensures deployed objects are pinned to exact versions and cached in Snowflake
- Snowflake scans Container Runtime images **daily** for CVEs; high/critical are patched within 30 days

---

## 9. What's Next

| Capability | What It Solves | Your Pain Today |
|-----------|---------------|------------------|
| **Model Registry** | Version, deploy, and track models as first-class objects | `_B13` suffix naming, lots of model copies, no lineage |
| **Cortex Analyst** | Natural language queries with a governed semantic layer | AskData has no semantic layer or governance |
| **dbt + Dynamic Tables** | Declarative SQL pipelines with always-fresh materialization | Manual orchestration, Credit Flow complexity |

### Recap

- **Dynamic Tables** — source tables update, features auto-refresh. No orchestrator. Replaces the data-prep portion of Credit Flow.
- **Notebook experience** — instant startup, persistent (no 14-day deletion), managed packages
- **Dependency management** — pip resolves cascading version changes automatically; Account Usage gives you blast-radius queries
- **Git integration** — native, 3 SQL statements. No GitHub Desktop, no CLI.
- **Private repo** — Artifactory integration for packages, PrivateLink for security

---
## Cleanup

Uncomment and run to drop all demo objects.

In [ ]:
%%sql -r cleanup
-- DROP DYNAMIC TABLE IF EXISTS demo_ian.valuations.acquisition_summary;
-- DROP DYNAMIC TABLE IF EXISTS demo_ian.valuations.feature_store;
-- DROP TABLE IF EXISTS demo_ian.valuations.credit_bureau_scores;
-- DROP TABLE IF EXISTS demo_ian.valuations.transaction_history;
-- DROP TABLE IF EXISTS demo_ian.valuations.account_demographics;
-- DROP SCHEMA IF EXISTS demo_ian.valuations;
-- DROP DATABASE IF EXISTS demo_ian;
SELECT 'Uncomment the lines above to clean up' AS status